In [6]:
from gel2d import gel2d, SolveNonlinearMinProblem
from ngsolve import *
import numpy as np
import datetime

class Solve_gel_bonded:
    def __init__(self, gel, order=3):           # corner_refinement capability not yet activated
        self.gel = gel
        self.order = order

    def add_mesh(self, mesh_file):
        self.mesh = Mesh(mesh_file)

    def Space(self):
        # Finite element space with zero-displacement boundary condition on the whole interface
        self.fes = VectorH1(self.mesh, order=self.order, \
                            dirichlet="bonded_interface|debonded_interface")
        print('nDoF = {}'.format(self.fes.ndof))

    def model(self):
        u = self.fes.TrialFunction()
        I = Id(self.mesh.dim)
        F = I + Grad(u)
        def negpart(var):
            #return max(-var,0)
            return (sqrt(var**2)-var)*0.5        
        # hydrogel model        
        AA = 1e5
        self.a = BilinearForm(self.fes, symmetric=False)
        self.a += Variation(  self.gel.W(F).Compile() * dx)
        self.a += Variation(  AA*negpart(y+u[1])**2 * dx)

def compute_solution(solver, mu_bar, tol=1e-08, maxits=50, alpha=5e-2, Draw_solution = True, filename = 'gridfunctions/result_gelsMu2D'):
    solver.gel.mu_bar.Set(mu_bar)
    solver.model()
    solver.gfu, _,_ = SolveNonlinearMinProblem(a= solver.a, gfu = solver.gfu,\
            FreeDofs =solver.fes.FreeDofs(), maxits=maxits, tol=tol)
    solver.gfu.Save(filename +'.gfu')
    # Now print status
    gfu = solver.gfu; mesh = solver.mesh; gel = solver.gel
    F=Id(2)+Grad(gfu)
    print('Average energy density = ', round(Integrate(gel.W(F), mesh)/(L*d),5))   
    vertex_values = [(y+gfu[1])(mesh(*p)) for p in mesh.ngmesh.Points()]
    print(f'Maximum vertical stretch: ', max(vertex_values)/gel.d)
    if Draw_solution:
        Draw((y+gfu[1])/d, mesh, deformation=gfu)

### MAIN ###
simulation_name = 'simulation01'
nIterations = 3
# data = ['dummy', '90', '1.62', '3', '0.2', '0.002']
data = ['dummy', '90', '1.62', '3', '0.2', '0.005']
# data = ['dummy', '90', '1.62', '3', '0.2', '0.004']
# data = ['dummy', '90', '1.62', '3', '0.2', '0.0055']
# data = ['dummy', '90', '1.62', '2', '0.2', '0.004']
# data = ['dummy', '90', '1.62', '2', '0.2', '0.005']
L = float(data[1])
d = float(data[2])
order = int(data[3])
phi0 = float(data[4])
mu_input = float(data[5])    
mu_bar = -mu_input  # internal sign convention
print(f'L={L} [mm], d={d} [mm], order={order}, phi0={phi0}, mu_bar={mu_bar}')
#
gel = gel2d(L,d,phi0,mu_bar)

modelling = Solve_gel_bonded (gel, order=order)
mesh_file = 'meshes1_62/mesh51.vol'
modelling.add_mesh(mesh_file)

solver = modelling
solver.Space()
solver.model()
solver.gfu = GridFunction(solver.fes)
solver.gfu.vec[:] = 0

mu_bar_end = solver.gel.mu_bar.Get()
print(f'mu_bar_end = {mu_bar_end}')
lambda_initial = 1.03
lambda_list = np.linspace(lambda_initial, solver.gel.lambda_target, nIterations+1, endpoint = True)
mu_bar_list = [solver.gel.mu_fun(la) for la in lambda_list[:-1]]
mu_bar_list.append(mu_bar_end)
indexes_iterations = range(nIterations+1)
for numIteration in indexes_iterations:
    lambda_i = lambda_list[numIteration]
    mu_bar_i = mu_bar_list[numIteration]
    print(f'Iteration {numIteration}: lambda={lambda_i}, mu_bar={mu_bar_i}')
# now save this list in a text file
filename = f'{simulation_name}/lambda_uniaxial_mu_bar_list.txt'
with open(filename, 'w') as f:
    f.write(f'nIterations = {nIterations}\n')
    f.write('lambda_uniaxial, mu_bar\n')
    for la, mu in zip(lambda_list, mu_bar_list):
        f.write(f'{la}, {mu}\n')

indexes_iterations = range(nIterations+1)
for numIteration in indexes_iterations:
    lambda_i = lambda_list[numIteration]
    mu_bar_i = mu_bar_list[numIteration]
    print(f'Iteration {numIteration}: lambda_uniaxial={lambda_i}, mu_bar={mu_bar_i}')
    tol = 1e-3; maxits = 100;
    filename = f'{simulation_name}/result_gelsMu2D' + solver.gel.filename_suffix + '_iter=' + str(numIteration).zfill(2)
    print(f'filename={filename}')
    compute_solution(solver, mu_bar_i, tol=tol, maxits=maxits, alpha=5e-2, Draw_solution = False, filename = filename)


L=90.0 [mm], d=1.62 [mm], order=3, phi0=0.2, mu_bar=-0.005
Entropic unit = 137.21 [MPa], G = 0.1372 [MPa], chi = 0.4, p0_bar = 0.0000233, lambda_target = 1.1298930828494025
gel.Gfun(lambda_target) = 0.13721349978427086
filename_suffix=_phi0=0.2_absMuBar=0.005
nDoF = 16670
mu_bar_end = -0.005
Iteration 0: lambda=1.03, mu_bar=-0.00654964107129134
Iteration 1: lambda=1.0632976942831343, mu_bar=-0.005978825687333496
Iteration 2: lambda=1.0965953885662683, mu_bar=-0.005464815271695122
Iteration 3: lambda=1.1298930828494025, mu_bar=-0.005
Iteration 0: lambda_uniaxial=1.03, mu_bar=-0.00654964107129134
filename=simulation01/result_gelsMu2D_phi0=0.2_absMuBar=0.005_iter=00
Average energy density =  -14.99264
Maximum vertical stretch:  1.029789983169423
Iteration 1: lambda_uniaxial=1.0632976942831343, mu_bar=-0.005978825687333496
filename=simulation01/result_gelsMu2D_phi0=0.2_absMuBar=0.005_iter=01
Average energy density =  -15.05894
Maximum vertical stretch:  1.0630736112834513
Iteration 2: lamb

In [7]:
from ngsolve.webgui import Draw
gfu = GridFunction(solver.fes)
gfu.Load(f'simulation02/result_gelsMu2D_phi0=0.2_absMuBar=0.005_iter=04.gfu')
mesh = solver.mesh; gel = solver.gel
F=Id(2)+Grad(gfu)
print('Average energy density = ', round(Integrate(gel.W(F), mesh)/(L*d),5))   
vertex_values = [(y+gfu[1])(mesh(*p)) for p in mesh.ngmesh.Points()]
print(f'Maximum vertical stretch: ', max(vertex_values)/gel.d)
Draw((y+gfu[1])/d, mesh, deformation=gfu)

Average energy density =  -15.17925
Maximum vertical stretch:  1.129633989005762


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene